## OS-Level Action을 사용하는 AgentCore Browser Tool

이 예제에서는 Amazon Bedrock AgentCore Browser에서 **OS-level action**(`InvokeBrowser`)을 사용하는 방법을 알아봅니다.

OS-level action을 사용하면 CDP/Playwright 자동화 계층을 완전히 우회해 브라우저 sandbox에서 mouse, keyboard, screenshot 및 scroll 작업을 직접 수행할 수 있습니다. 다음 대상과 상호 작용할 때 유용합니다.

- **OS-native dialog**(file upload/download prompt, print dialog, 인증 pop-up)
- **Browser chrome element**(address bar, extension popup, 권한 banner)
- CDP 기반 자동화에서 OS로 전송할 수 없는 **keyboard shortcut**(Ctrl+S, Ctrl+A, Alt+Tab)
- DOM selector가 없는 **Canvas / WebGL content**
- CDP 기반 자동화로 처리하기 어려운 **모든 element**

이 Notebook에서는 SigV4-signed REST 호출로 `InvokeBrowser` API를 사용하는 과정을 살펴봅니다. Mouse action(click, move, drag), scroll, keyboard 입력(type, press, shortcut), screenshot을 다룹니다.

### 1. Dependency 설치

In [ ]:
!uv pip install -qU -r requirements.txt

import boto3

print("botocore:", boto3.__version__)

### 2. AWS 자격 증명 불러오기

실습을 위해 terminal에서 자격 증명 file을 생성합니다.

In [ ]:
import os
from pathlib import Path

env_file = Path(".env")
assert env_file.exists(), "Missing .env file. Run: ./setup_aws_creds.sh <account_id>"

for line in env_file.read_text().splitlines():
    if "=" in line and not line.startswith("#"):
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip()

assert os.environ.get("AWS_ACCESS_KEY_ID"), "Failed to load credentials from .env"
print("AWS credentials loaded from .env")

### 3. 설정

변수를 초기화합니다.

In [ ]:
import boto3

browser_boto3 = boto3.client(
    "bedrock-agentcore-control",
    region_name="us-west-2",
)

BROWSER_NAME = "browser_with_os_actions"

Python path에 helpers 디렉터리를 추가합니다.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

#### 3.1 IAM Role 생성

그런 다음 AgentCore Browser에 연결할 사용자 지정 IAM role을 생성합니다.
OS-level action에 필요한 추가 권한 `bedrock-agentcore:InvokeBrowser`에 유의하세요.

In [ ]:
# IAM ROLE 생성
from helpers.utils import create_agentcore_execution_role, SAMPLE_ROLE_NAME

execution_role_arn = create_agentcore_execution_role(SAMPLE_ROLE_NAME)

IAM role이 전파되도록 10초 동안 기다립니다.

In [ ]:
import time

time.sleep(10)

### 4. 사용자 지정 AgentCore Browser 생성

이 예제에서는 녹화가 활성화된 사용자 지정 브라우저를 생성합니다.

In [ ]:
created_browser = browser_boto3.create_browser(
    name=BROWSER_NAME,
    executionRoleArn=execution_role_arn,
    networkConfiguration={"networkMode": "PUBLIC"},
)

browser_id = created_browser["browserId"]
print(f"Browser ID: {browser_id}")

#### 5. OS-level action이 활성화된 브라우저 세션 시작


In [ ]:
from helpers.browser import get_credentials, invoke, start_session, stop_session

creds, default_region = get_credentials()
BEDROCK_AGENTCORE_DP_ENDPOINT = f"https://bedrock-agentcore.{default_region}.amazonaws.com/"

In [ ]:
creds, default_region

In [ ]:
import time

sid = start_session(BEDROCK_AGENTCORE_DP_ENDPOINT, browser_id, region=default_region, credentials=creds)
print("  Waiting 3s for session to initialize...")

time.sleep(3)

In [ ]:
print("\n── Happy Path: Mouse Actions ──")

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseClick": {"x": 600, "y": 370, "button": "LEFT"}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse click status: {r.status_code}, action" {r.json()["result"]}')

In [ ]:
r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseClick": {"x": 500, "y": 300, "button": "LEFT", "clickCount": 2}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse click status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseClick": {"x": 200, "y": 400, "button": "RIGHT", "clickCount": 1}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse click status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseClick": {"x": 960, "y": 540, "button": "MIDDLE", "clickCount": 1}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse click status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseClick": {"x": 500, "y": 300, "button": "LEFT", "clickCount": 10}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse click status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseMove": {"x": 800, "y": 600}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse click status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseMove": {"x": 1, "y": 1}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse click status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {
        "mouseDrag": {
            "startX": 1,
            "startY": 1,
            "endX": 705,
            "endY": 180,
            "button": "LEFT",
        }
    },
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse click status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {
        "mouseDrag": {
            "startX": 1,
            "startY": 1,
            "endX": 370,
            "endY": 330,
            "button": "LEFT",
        }
    },
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse click status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {
        "mouseDrag": {
            "startX": 500,
            "startY": 300,
            "endX": 100,
            "endY": 200,
            "button": "MIDDLE",
        }
    },
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse click status: {r.status_code}, action" {r.json()["result"]}')

In [ ]:
print("\n── Happy Path: Scroll Actions ──")

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseScroll": {"x": 800, "y": 600, "deltaX": 0, "deltaY": -500}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse action status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseScroll": {"x": 500, "y": 300, "deltaX": 300, "deltaY": 0}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse action status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseScroll": {"x": 500, "y": 300, "deltaX": -100, "deltaY": -200}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse action status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"mouseScroll": {"x": 500, "y": 300, "deltaX": 1000, "deltaY": 1000}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Mouse action status: {r.status_code}, action" {r.json()["result"]}')

In [ ]:
print("\n── Happy Path: Keyboard Actions ──")

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyType": {"text": "Hello World"}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyType": {"text": "user@example.com!#$%^&*()"}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyType": {"text": "https://www.example.com"}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyType": {"text": "1" * 10000}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyPress": {"key": "enter"}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyPress": {"key": "tab"}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyPress": {"key": "escape"}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyPress": {"key": "backspace", "presses": 5}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyPress": {"key": "ArrowDown", "presses": 100}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyShortcut": {"keys": ["ctrl", "s"]}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyShortcut": {"keys": ["ctrl", "p"]}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"keyShortcut": {"keys": ["ctrl", "shift", "i"]}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Keyboard status: {r.status_code}, action" {r.json()["result"]}')

In [ ]:
from IPython.display import Image, display
import base64


def check_screenshot(resp):
    data = resp.json().get("result", {}).get("screenshot", {}).get("data")
    if not data:
        return "screenshot data is empty"

    img_bytes = base64.b64decode(data)
    display(Image(img_bytes))


print("\n── Happy Path: Screenshot ──")

r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"screenshot": {"format": "PNG"}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Screenshot status: {r.status_code}, action" {r.json()["result"]}')
check_screenshot(r)

In [ ]:
r = invoke(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    {"screenshot": {}},
    region=default_region,
    credentials=creds,
    browser_id=browser_id,
)
print(f'Screenshot status: {r.status_code}, action" {r.json()["result"]}')
check_screenshot(r)

#### 6. 브라우저 세션 중지

In [ ]:
print("\n── Teardown: Stopping browser session ──")
stop_session(
    BEDROCK_AGENTCORE_DP_ENDPOINT,
    sid,
    browser_id,
    region=default_region,
    credentials=creds,
)

### 7. 정리(선택 사항)

사용자 지정 브라우저를 삭제하고 IAM role을 제거합니다.

#### 7.1 사용자 지정 브라우저 삭제

In [ ]:
browser_boto3.delete_browser(browserId=browser_id)
print(f"Browser {browser_id} deleted")

#### 7.2 IAM role 삭제

In [ ]:
from helpers.utils import delete_agentcore_execution_role, SAMPLE_ROLE_NAME

response = delete_agentcore_execution_role(SAMPLE_ROLE_NAME)

print(f"Role Deleted: {response}")